In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Each notebook must load its own tables. We only need 4 files here.
inv      = pd.read_csv("../Data/Raw/Inventory_History.csv", encoding="utf-8-sig")
sales    = pd.read_csv("../Data/Raw/Sales_Transactions.csv", encoding="utf-8-sig")
qual     = pd.read_csv("../Data/Raw/Quality.csv", encoding="utf-8-sig")
printing = pd.read_csv("../Data/Raw/Printing_Batches.csv", encoding="utf-8-sig")

print("inv rows:", len(inv))
print("sales rows:", len(sales))
print("qual rows:", len(qual))
print("printing rows:", len(printing))

inv rows: 6372
sales rows: 180000
qual rows: 9000
printing rows: 1900


In [5]:

# Target: did this product-location-month suffer a stockout?
inv["Had_Stockout"] = (inv["Stockout_Days_In_Month"] > 0).astype(int)
print("Stockout rate:", inv["Had_Stockout"].mean().round(3))

features = ["Opening_Stock", "Received_Qty", "Sold_Qty", "Returned_Qty",
            "Avg_Daily_Sales_90D", "Safety_Stock_Units", "Reorder_Point_Units",
            "Supplier_Lead_Time_Days", "Days_of_Cover"]

X = inv[features]
y = inv["Had_Stockout"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

# Dumb baseline
dumb = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

# Real model
rf = RandomForestClassifier(n_estimators=100, max_depth=6,
                            random_state=42, class_weight="balanced").fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("DUMB  ROC-AUC:", round(roc_auc_score(y_test, dumb.predict_proba(X_test)[:, 1]), 3))
print("MODEL ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 3))
print("MODEL Recall:", round(recall_score(y_test, rf_pred), 3))
print("MODEL Precision:", round(precision_score(y_test, rf_pred), 3))
print("\nWhat drives stockouts:")
print(pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False).round(3))


# ============ TEST 2: Quality-return risk (the feedback loop!) ============
# Give each product its factory defect rate, then see if it predicts customer returns
defect_rate = (qual.groupby("Product_ID")["Rejected_Qty"].sum()
               / qual.groupby("Product_ID")["Inspected_Qty"].sum()).rename("Product_Defect_Rate")

s2 = sales.merge(defect_rate, on="Product_ID", how="left")
s2["Target"] = (s2["Quality_Related_Return"] == "Yes").astype(int)

features2 = ["Discount_Pct", "List_Price_EGP", "Quantity", "Product_Defect_Rate"]
X2 = s2[features2]
y2 = s2["Target"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2,
                                                        random_state=42, stratify=y2)

dumb2 = DummyClassifier(strategy="most_frequent").fit(X2_train, y2_train)
rf2 = RandomForestClassifier(n_estimators=100, max_depth=6,
                             random_state=42, class_weight="balanced").fit(X2_train, y2_train)
rf2_pred = rf2.predict(X2_test)
rf2_prob = rf2.predict_proba(X2_test)[:, 1]

print("\nDUMB  ROC-AUC:", round(roc_auc_score(y2_test, dumb2.predict_proba(X2_test)[:, 1]), 3))
print("MODEL ROC-AUC:", round(roc_auc_score(y2_test, rf2_prob), 3))
print("MODEL Recall:", round(recall_score(y2_test, rf2_pred), 3))
print("MODEL Precision:", round(precision_score(y2_test, rf2_pred), 3))
print("\nWhat drives quality returns:")
print(pd.Series(rf2.feature_importances_, index=features2).sort_values(ascending=False).round(3))

Stockout rate: 0.023
DUMB  ROC-AUC: 0.5
MODEL ROC-AUC: 1.0
MODEL Recall: 1.0
MODEL Precision: 0.967

What drives stockouts:
Days_of_Cover              0.478
Opening_Stock              0.293
Avg_Daily_Sales_90D        0.073
Sold_Qty                   0.051
Safety_Stock_Units         0.050
Reorder_Point_Units        0.037
Received_Qty               0.010
Returned_Qty               0.005
Supplier_Lead_Time_Days    0.002
dtype: float64

DUMB  ROC-AUC: 0.5
MODEL ROC-AUC: 0.58
MODEL Recall: 0.876
MODEL Precision: 0.017

What drives quality returns:
Discount_Pct           0.519
Quantity               0.257
Product_Defect_Rate    0.154
List_Price_EGP         0.070
dtype: float64


In [7]:
# 1) Pull the simple human rule out of the hired model
no_stock = inv[inv["Had_Stockout"] == 0]["Days_of_Cover"]
yes_stock = inv[inv["Had_Stockout"] == 1]["Days_of_Cover"]

print("Avg Days_of_Cover in NORMAL months:", round(no_stock.mean(), 1))
print("Avg Days_of_Cover in STOCKOUT months:", round(yes_stock.mean(), 1))

# 2) Save the hired model so the future app can use it
import joblib
from pathlib import Path
Path("../models").mkdir(exist_ok=True)
joblib.dump(rf, "../models/stockout_model.pkl")
print("Saved: models/stockout_model.pkl")

Avg Days_of_Cover in NORMAL months: 343.2
Avg Days_of_Cover in STOCKOUT months: 0.0
Saved: models/stockout_model.pkl
